# AI-Text Detector Robustness: RAID + MAGE
This version drops the live paraphrase / adaptive attack for now and focuses on one question: **can the static RAID attacks fool a classifier that was trained only on clean text, and does adversarial retraining fix it?**

**The Workflow:**
1. **Data Prep:** Read the *whole* RAID `train.csv` once and keep a uniform random sample (so every attack type and domain is represented) -> Parquet -> PySpark (clean, balance, split by `source_id`) + diagnostics.
2. **Zero-Shot Baseline:** GPT-2 perplexity on every static RAID attack (full AUROC / TPR / FPR tables + slices).
3. **Naive Classifier:** trained **only on clean (`none`) text**, evaluated on *every* attack. This is the one we try to fool.
4. **Robust Defenders:** (a) retrained on clean + 4 attacks with paraphrase held out, (b) retrained on all attacks.
5. **Final Comparison:** all models side by side, per attack.
6. **Generalization Check:** MAGE out-of-distribution data (nobody trained on it), with TPR / FPR / AUROC for every model.



In [1]:
!pip -q install pyspark datasets transformers accelerate scikit-learn pyarrow huggingface_hub

In [2]:
import os, time, numpy as np, pandas as pd
import torch
from IPython.display import display
from pyspark.sql import SparkSession, functions as F
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import roc_auc_score

# Suppress Hugging Face warnings for cleaner outputs
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

DATA_DIR = "data"; os.makedirs(DATA_DIR, exist_ok=True)
SEED = 42
RAW_PARQUET = f"{DATA_DIR}/raid_raw_v4.parquet"
CLEAN_PARQUET = f"{DATA_DIR}/raid_clean_v4"
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", dev)

Device: cuda


## 1. Prepare the RAID Dataset

In [3]:
COLS = ["id","adv_source_id","source_id","model","decoding","repetition_penalty",
        "attack","domain","title","prompt","generation"]

# Keep-probabilities per row while scanning the full train.csv (human rows are rare, so keep more of them).
# Raise them for a bigger sample; Spark balances human vs AI afterwards anyway.
P_HUMAN, P_AI = 0.5, 0.012

def sample_raid_csv(path, p_human, p_ai, seed=SEED, chunk=50_000, usecols=COLS):
    """One pass over the whole CSV, keeping a uniform random subset -> all attacks/domains/models covered."""
    rng = np.random.default_rng(seed); keep = []; scanned = 0; t0 = time.time()
    for ch in pd.read_csv(path, usecols=usecols, dtype=str, keep_default_na=False, chunksize=chunk):
        p = np.where((ch["model"] == "human").to_numpy(), p_human, p_ai)
        keep.append(ch[rng.random(len(ch)) < p])
        scanned += len(ch)
        if (scanned // chunk) % 20 == 0:
            print(f"  scanned {scanned:,} rows | kept {sum(map(len, keep)):,} | {time.time()-t0:.0f}s")
    return pd.concat(keep, ignore_index=True)

In [4]:
if not os.path.exists(RAW_PARQUET):
    from huggingface_hub import hf_hub_download, list_repo_files
    cands = [f for f in list_repo_files("liamdugan/raid", repo_type="dataset") if os.path.basename(f) == "train.csv"]
    assert cands, "train.csv not found in liamdugan/raid - check the repo file list"
    print("First run only: downloading ~12 GB and sampling (can take a while):", cands[0])
    path = hf_hub_download("liamdugan/raid", cands[0], repo_type="dataset")
    sample_raid_csv(path, P_HUMAN, P_AI).to_parquet(RAW_PARQUET, index=False)

raw = pd.read_parquet(RAW_PARQUET)
print("Raw sample shape:", raw.shape)
print("\nRows per attack (all 11 attacks + none should appear):")
display(raw.attack.value_counts().rename_axis("attack").to_frame("rows"))
print("Rows per domain:")
display(raw.domain.value_counts().rename_axis("domain").to_frame("rows"))
print("Human vs AI rows:", dict(raw.model.eq("human").map({True: "human", False: "AI"}).value_counts()))
del raw

First run only: downloading ~12 GB and sampling (can take a while): train.csv


train.csv: reconstructing file:   0%|          |  0.00B / 11.8GB            

train.csv: downloading bytes:           |  0.00B            

  scanned 1,000,000 rows | kept 31,917 | 34s
  scanned 2,000,000 rows | kept 54,272 | 64s
  scanned 3,000,000 rows | kept 78,456 | 94s
  scanned 4,000,000 rows | kept 109,782 | 122s
  scanned 5,000,000 rows | kept 132,971 | 152s
Raw sample shape: (145677, 11)

Rows per attack (all 11 attacks + none should appear):


,rows
attack,
synonym,12269
none,12222
article_deletion,12189
zero_width_space,12188
homoglyph,12172
number,12170
perplexity_misspelling,12119
paraphrase,12117
whitespace,12101


Rows per domain:


,rows
domain,
poetry,19494
news,19437
books,19406
recipes,19403
wiki,19337
reddit,19249
abstracts,19141
reviews,10210


Human vs AI rows: {'human': np.int64(80312), 'AI': np.int64(65365)}


In [5]:
spark = (SparkSession.builder.master("local[*]").appName("raid-robustness")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "32").getOrCreate())

df = spark.read.parquet(RAW_PARQUET)
df = (df.withColumn("label", F.when(F.col("model") == "human", 0).otherwise(1))   # 1 = AI
        .withColumn("text", F.trim(F.col("generation")))
        .withColumn("n_words", F.size(F.split("text", r"\s+")))
        .filter(F.col("n_words") >= 30)
        .dropDuplicates(["id"])
        .dropDuplicates(["text"]))

# Balance human vs AI (down-sample whichever class is larger)
c = {r["label"]: r["count"] for r in df.groupBy("label").count().collect()}
frac = {0: min(1.0, c[1] / c[0]), 1: min(1.0, c[0] / c[1])}
df = df.sampleBy("label", frac, seed=SEED)

# Group split by source document to prevent train/test leakage
df = df.withColumn("bucket", F.abs(F.hash("source_id")) % 100)
df = df.withColumn("split", F.when(F.col("bucket") < 70, "train")
                             .when(F.col("bucket") < 85, "val").otherwise("test"))

df.drop("bucket").write.mode("overwrite").partitionBy("split").parquet(CLEAN_PARQUET)
df = spark.read.parquet(CLEAN_PARQUET)
print("Cleaned dataset count:", df.count())
df.groupBy("split", "label").count().orderBy("split", "label").show()
print("Rows per attack and class (0 = human, 1 = AI):")
display(df.groupBy("attack").pivot("label").count().orderBy("attack").toPandas().rename(columns={"0": "human", "1": "AI"}))

Cleaned dataset count: 129725
+-----+-----+-----+
|split|label|count|
+-----+-----+-----+
| test|    0| 9817|
| test|    1| 9825|
|train|    0|45302|
|train|    1|45210|
|  val|    0| 9815|
|  val|    1| 9756|
+-----+-----+-----+

Rows per attack and class (0 = human, 1 = AI):


,attack,human,AI
0,alternative_spelling,4999,5275
1,article_deletion,5695,5386
2,homoglyph,5723,5409
3,insert_paragraphs,5642,5301
4,none,4503,5383
5,number,4856,5391
6,paraphrase,5694,5358
7,perplexity_misspelling,5151,5423
8,synonym,5727,5538
9,upper_lower,5664,5382


## 2. Evaluate Zero-Shot GPT-2 Baseline
First the shared reporting helpers (used by every model), then GPT-2 perplexity scored on the **static RAID attacks**.

In [6]:
def calibrate(val_pdf, fpr=0.05):
    """Threshold = (1-fpr) quantile of clean human scores -> 5% FPR on clean human text."""
    h = val_pdf[(val_pdf.label == 0) & (val_pdf.attack == "none")].score
    return float(np.quantile(h, 1 - fpr))

def report(pdf, thr, by="attack", seen=None):
    rows = []
    for k, g in pdf.groupby(by):
        h, a = g[g.label == 0].score, g[g.label == 1].score
        auc = roc_auc_score(g.label, g.score) if g.label.nunique() == 2 else np.nan
        rows.append({by: k, "n": len(g), "AUROC": auc,
                     "TPR@thr": (a > thr).mean() if len(a) else np.nan,
                     "FPR@thr": (h > thr).mean() if len(h) else np.nan})
    out = pd.DataFrame(rows).sort_values("AUROC").reset_index(drop=True)
    if seen is not None and by == "attack":
        out["seen_in_training"] = out["attack"].isin(seen)
    return out

def report_by_model(pdf, thr):
    """Generator slice. A generator's rows are all AI, so AUROC is computed against the human rows."""
    h = pdf[pdf.label == 0]
    rows = [{"model": "human", "n": len(h), "AUROC": np.nan, "TPR@thr": np.nan,
             "FPR@thr": (h.score > thr).mean() if len(h) else np.nan}]
    for m, g in pdf[pdf.label == 1].groupby("model"):
        auc = (roc_auc_score(np.r_[np.zeros(len(h)), np.ones(len(g))], np.r_[h.score, g.score])
               if len(h) and len(g) else np.nan)
        rows.append({"model": m, "n": len(g), "AUROC": auc,
                     "TPR@thr": (g.score > thr).mean(), "FPR@thr": np.nan})
    return pd.DataFrame(rows)

def full_report(name, test_pdf, thr, seen=None):
    print(f"==================== {name} ====================")
    print(f"5% FPR threshold: {thr:.4f}\n")
    print(f"--- {name}: static RAID attacks (full TPR/FPR table) ---")
    rep = report(test_pdf, thr, "attack", seen)
    display(rep.round(4))
    print(f"\n--- {name}: slice by generator (clean only) ---")
    display(report_by_model(test_pdf[test_pdf.attack == "none"], thr).round(4))
    print(f"\n--- {name}: slice by domain (all attacks) ---")
    display(report(test_pdf, thr, "domain").round(4))
    return rep

def side_by_side(reports, sort_by):
    parts = []
    for name, r in reports.items():
        r = r.set_index("attack")[["AUROC", "TPR@thr", "FPR@thr"]].copy()
        r.columns = pd.MultiIndex.from_product([[name], r.columns])
        parts.append(r)
    out = pd.concat(parts, axis=1)
    return out.sort_values((sort_by, "AUROC"))

In [7]:
def nll_texts(texts, tok, lm, dev, max_len=256, bs=16):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], return_tensors="pt", padding=True, truncation=True, max_length=max_len).to(dev)
        with torch.no_grad():
            logits = lm(**enc).logits
        lp = torch.log_softmax(logits[:, :-1].float(), -1)
        tgt = enc.input_ids[:, 1:]
        tl = lp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
        m = enc.attention_mask[:, 1:].float()
        out += (-(tl * m).sum(1) / m.sum(1).clamp(min=1)).cpu().tolist()
    return out

def ppl_partition(iterator):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained("gpt2"); tok.pad_token = tok.eos_token
    lm = AutoModelForCausalLM.from_pretrained("gpt2").to(dev).eval()
    for pdf in iterator:
        nll = nll_texts(pdf["text"].tolist(), tok, lm, dev)
        yield pdf[["id"]].assign(score=[-x for x in nll])

def score_split(split, n=None, parts=1, clean_only=False):
    base = df.filter(F.col("split") == split)
    if clean_only: base = base.filter(F.col("attack") == "none")     # calibration only needs clean text
    sub = base.select("id", "text")
    if n: sub = sub.orderBy(F.rand(SEED)).limit(n)                   # random sample, not the first n rows
    scored = sub.coalesce(parts).mapInPandas(ppl_partition, schema="id string, score double")
    meta = base.select("id", "label", "attack", "model", "domain", "n_words")
    return scored.join(meta, "id").toPandas()

# GPT-2 for scoring on the driver (used for MAGE later)
g_tok = AutoTokenizer.from_pretrained("gpt2"); g_tok.pad_token = g_tok.eos_token
g_lm = AutoModelForCausalLM.from_pretrained("gpt2").to(dev).eval()
def detector_score(texts): return [-x for x in nll_texts(texts, g_tok, g_lm, dev)]

GPT2_NAME = "GPT-2 zero-shot"
t0 = time.time()
val_scores  = score_split("val",  n=10_000, clean_only=True)
test_scores = score_split("test", n=20_000)
print(f"GPT-2 Scoring time (s): {time.time() - t0:.1f}")

thr_gpt2 = calibrate(val_scores)
rep_gpt2 = full_report(GPT2_NAME, test_scores, thr_gpt2)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

... (5 outputs left out)


--- GPT-2 zero-shot: slice by generator (clean only) ---


,model,n,AUROC,TPR@thr,FPR@thr
0,human,706,NaN,NaN,0.0425
1,chatgpt,33,0.8150,0.3030,NaN
2,cohere,42,0.8510,0.3333,NaN
3,cohere-chat,49,0.8127,0.4082,NaN
4,gpt2,87,0.8288,0.4598,NaN
5,gpt3,49,0.9240,0.5102,NaN
6,gpt4,48,0.7729,0.3542,NaN
7,llama-chat,95,0.8342,0.4211,NaN
8,mistral,106,0.6992,0.3208,NaN
9,mistral-chat,91,0.8442,0.5165,NaN



--- GPT-2 zero-shot: slice by domain (all attacks) ---


,domain,n,AUROC,TPR@thr,FPR@thr
0,poetry,2299,0.6451,0.2221,0.1040
1,recipes,2821,0.6944,0.6052,0.1907
2,wiki,2701,0.7126,0.3173,0.0985
3,news,2618,0.7218,0.3095,0.0796
4,reviews,1423,0.7510,0.1911,0.0743
5,abstracts,2818,0.7528,0.1645,0.0832
6,reddit,2626,0.7560,0.2929,0.0848
7,books,2336,0.7579,0.2172,0.0905


## 3. Naive Baseline (Classifier trained on clean text only)
Train **only on clean (`none`) text**, then evaluate on *every* RAID attack. This is the model we are trying to fool: it has never seen a single attacked example.

`CLF_NAME` picks the architecture. `"roberta-base"` is the default; set it to `"gpt2"` to fine-tune GPT-2 as a supervised classifier instead (everything else stays the same).

In [8]:
CLF_NAME = "roberta-base"          # or "gpt2"
NAIVE_NAME, ROBUST_NAME, ROBUST_ALL_NAME = f"Naive {CLF_NAME}", f"Robust {CLF_NAME}", f"Robust-all {CLF_NAME}"

c_tok = AutoTokenizer.from_pretrained(CLF_NAME)
if c_tok.pad_token is None: c_tok.pad_token = c_tok.eos_token
def enc(b): return c_tok(b["text"], truncation=True, max_length=256)

def roberta_scores(model, texts, bs=32):
    model.eval().to(dev); out = []
    for i in range(0, len(texts), bs):
        e = c_tok(texts[i:i+bs], truncation=True, max_length=256, padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            out += torch.softmax(model(**e).logits, -1)[:, 1].cpu().tolist()
    return out

def score_with(model, split, n, clean_only=False):
    base = spark.read.parquet(CLEAN_PARQUET).filter(F.col("split") == split)
    if clean_only: base = base.filter(F.col("attack") == "none")
    pdf = (base.orderBy(F.rand(SEED)).select("id", "text", "label", "attack", "model", "domain")
           .limit(n).toPandas())
    pdf["score"] = roberta_scores(model, pdf.text.tolist())
    return pdf

def new_clf():
    m = AutoModelForSequenceClassification.from_pretrained(CLF_NAME, num_labels=2)
    m.config.pad_token_id = c_tok.pad_token_id
    return m

def train_clf(train_pdf, out_dir):
    ds = Dataset.from_pandas(train_pdf, preserve_index=False).map(enc, batched=True)
    model = new_clf()
    a = TrainingArguments(out_dir, per_device_train_batch_size=16, num_train_epochs=1,
                          learning_rate=2e-5, fp16=torch.cuda.is_available(),
                          report_to="none", save_strategy="no", seed=SEED)
    tr = Trainer(model=model, args=a, train_dataset=ds, processing_class=c_tok,
                 data_collator=DataCollatorWithPadding(c_tok))
    tr.train(); del tr
    return model

# Same training budget for naive and robust, so the comparison is fair
n_clean_train = df.filter((F.col("split") == "train") & (F.col("attack") == "none")).count()
N_TRAIN = min(20_000, n_clean_train)
print(f"Clean training rows available: {n_clean_train:,} -> N_TRAIN = {N_TRAIN:,} for BOTH models")

# Train on CLEAN text only
tr_naive = (df.filter((F.col("split") == "train") & (F.col("attack") == "none"))
            .orderBy(F.rand(SEED)).select("text", "label").limit(N_TRAIN).toPandas())
print(f"Training {NAIVE_NAME} on {len(tr_naive):,} clean rows...")
clf_naive = train_clf(tr_naive, "ckpt_naive")

# Evaluate on every static RAID attack
rv_naive = score_with(clf_naive, "val", 10_000, clean_only=True)
rt_naive = score_with(clf_naive, "test", 20_000)
thr_naive = calibrate(rv_naive)
rep_naive = full_report(NAIVE_NAME + " (clean-only)", rt_naive, thr_naive, seen={"none"})

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Clean training rows available: 6,928 -> N_TRAIN = 6,928 for BOTH models
Training Naive roberta-base on 6,928 clean rows...


... (6 outputs left out)


--- Naive roberta-base (clean-only): slice by generator (clean only) ---


,model,n,AUROC,TPR@thr,FPR@thr
0,human,706,NaN,NaN,0.0581
1,chatgpt,33,1.0000,1.0000,NaN
2,cohere,42,0.9908,0.9286,NaN
3,cohere-chat,49,0.9998,1.0000,NaN
4,gpt2,87,0.9965,0.9885,NaN
5,gpt3,49,1.0000,1.0000,NaN
6,gpt4,48,0.9999,1.0000,NaN
7,llama-chat,95,1.0000,1.0000,NaN
8,mistral,106,0.9916,0.9623,NaN
9,mistral-chat,91,1.0000,1.0000,NaN



--- Naive roberta-base (clean-only): slice by domain (all attacks) ---


,domain,n,AUROC,TPR@thr,FPR@thr
0,reviews,1423,0.9191,0.9368,0.3301
1,reddit,2626,0.9264,0.9606,0.2710
2,books,2336,0.9316,0.9387,0.2580
3,news,2618,0.9355,0.9474,0.2595
4,wiki,2701,0.9382,0.9322,0.2451
5,poetry,2299,0.9527,0.9480,0.2613
6,abstracts,2818,0.9610,0.9840,0.2950
7,recipes,2821,0.9672,0.9451,0.1786


## 4. Robust Defenders (Adversarial Retraining)
Two retrained models (4a uses the **same number of training rows** as the naive one; 4b uses 3x, see below):
* **4a. Robust (partial):** clean + `homoglyph` + `zero_width_space` + `insert_paragraphs` (the attacks the naive model failed on). **`paraphrase` and the other attacks stay unseen**, so its paraphrase result is a real generalization test.
* **4b. Robust (all attacks):** trained on every attack, with **3x the rows** so each attack gets as many examples as in 4a (12 attack types vs 4). Its RAID numbers are in-distribution, so we do **not** use them as evidence of generalization; this is the model we carry into the final MAGE test.

The `seen_in_training` column shows which attacks each model saw.

In [9]:
# 4a. PARTIAL: clean + the attacks that broke the naive model; paraphrase etc. are held out
TRAIN_ATTACKS = ["none", "homoglyph", "zero_width_space", "insert_paragraphs"]
tr_robust = (df.filter((F.col("split") == "train") & F.col("attack").isin(TRAIN_ATTACKS))
             .orderBy(F.rand(SEED)).select("text", "label").limit(N_TRAIN).toPandas())
print(f"Training {ROBUST_NAME} on {len(tr_robust):,} rows (attacks: {TRAIN_ATTACKS})...")
clf_robust = train_clf(tr_robust, "ckpt_robust")

# Evaluate on every static RAID attack
rv_robust = score_with(clf_robust, "val", 10_000, clean_only=True)
rt_robust = score_with(clf_robust, "test", 20_000)
thr_robust = calibrate(rv_robust)
rep_robust = full_report(ROBUST_NAME + " (4 attacks, paraphrase held out)", rt_robust, thr_robust, seen=set(TRAIN_ATTACKS))

Training Robust roberta-base on 6,928 rows (attacks: ['none', 'homoglyph', 'zero_width_space', 'insert_paragraphs'])...


Map:   0%|          | 0/6928 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'train_runtime': '97.5', 'train_samples_per_second': '71.06', 'train_steps_per_second': '4.441', 'train_loss': '0.4945', 'epoch': '1'}
==================== Robust roberta-base (4 attacks, paraphrase held out) ====================
5% FPR threshold: 0.9670

--- Robust roberta-base (4 attacks, paraphrase held out): static RAID attacks (full TPR/FPR table) ---


,attack,n,AUROC,TPR@thr,FPR@thr,seen_in_training
0,zero_width_space,1669,0.6170,0.1802,0.0154,True
1,paraphrase,1662,0.8232,0.5916,0.1604,False
2,homoglyph,1708,0.8835,0.5680,0.0182,True
3,insert_paragraphs,1612,0.9720,0.8982,0.0521,True
4,upper_lower,1675,0.9838,0.9110,0.0378,False
5,whitespace,1682,0.9845,0.9193,0.0197,False
6,number,1592,0.9869,0.9274,0.0359,False
7,none,1476,0.9877,0.9325,0.0397,True
8,perplexity_misspelling,1601,0.9881,0.9347,0.0375,False
9,article_deletion,1671,0.9891,0.9397,0.0303,False



--- Robust roberta-base (4 attacks, paraphrase held out): slice by generator (clean only) ---


,model,n,AUROC,TPR@thr,FPR@thr
0,human,706,NaN,NaN,0.0397
1,chatgpt,33,0.9992,1.0000,NaN
2,cohere,42,0.9792,0.9048,NaN
3,cohere-chat,49,0.9970,0.9796,NaN
4,gpt2,87,0.9838,0.8966,NaN
5,gpt3,49,0.9999,1.0000,NaN
6,gpt4,48,0.9987,1.0000,NaN
7,llama-chat,95,0.9991,1.0000,NaN
8,mistral,106,0.9886,0.9528,NaN
9,mistral-chat,91,0.9997,1.0000,NaN



--- Robust roberta-base (4 attacks, paraphrase held out): slice by domain (all attacks) ---


,domain,n,AUROC,TPR@thr,FPR@thr
0,wiki,2701,0.9092,0.7294,0.0714
1,reviews,1423,0.9298,0.8103,0.1004
2,books,2336,0.9495,0.7968,0.0419
3,reddit,2626,0.9585,0.7944,0.0165
4,news,2618,0.9675,0.7730,0.0037
5,poetry,2299,0.9682,0.8114,0.0307
6,abstracts,2818,0.9720,0.8655,0.0759
7,recipes,2821,0.9869,0.8628,0.0179


In [10]:
# 4b. ALL ATTACKS: 3x the rows (same examples per attack as 4a), drawn at random from every attack type (incl. paraphrase)
ALL_ATTACKS = sorted(r["attack"] for r in df.select("attack").distinct().collect())
# 3x rows: 12 attack types vs 4 in the partial model -> each attack gets the same number of examples as in 4a
N_TRAIN_ALL = 3 * N_TRAIN
tr_all = (df.filter(F.col("split") == "train")
          .orderBy(F.rand(SEED)).select("text", "label").limit(N_TRAIN_ALL).toPandas())
print(f"Training {ROBUST_ALL_NAME} on {len(tr_all):,} rows (attacks: {ALL_ATTACKS})...")
clf_all = train_clf(tr_all, "ckpt_robust_all")

rv_all = score_with(clf_all, "val", 10_000, clean_only=True)
rt_all = score_with(clf_all, "test", 20_000)
thr_all = calibrate(rv_all)
rep_all = full_report(ROBUST_ALL_NAME + " (all attacks; RAID numbers are in-distribution)", rt_all, thr_all, seen=set(ALL_ATTACKS))

Training Robust-all roberta-base on 20,784 rows (attacks: ['alternative_spelling', 'article_deletion', 'homoglyph', 'insert_paragraphs', 'none', 'number', 'paraphrase', 'perplexity_misspelling', 'synonym', 'upper_lower', 'whitespace', 'zero_width_space'])...


Map:   0%|          | 0/20784 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'loss': '0.335', 'grad_norm': '47.22', 'learning_rate': '1.232e-05', 'epoch': '0.3849'}
{'loss': '0.2222', 'grad_norm': '10.01', 'learning_rate': '4.619e-06', 'epoch': '0.7698'}
{'train_runtime': '288.1', 'train_samples_per_second': '72.13', 'train_steps_per_second': '4.508', 'train_loss': '0.2541', 'epoch': '1'}
==================== Robust-all roberta-base (all attacks; RAID numbers are in-distribution) ====================
5% FPR threshold: 0.9388

--- Robust-all roberta-base (all attacks; RAID numbers are in-distribution): static RAID attacks (full TPR/FPR table) ---


,attack,n,AUROC,TPR@thr,FPR@thr,seen_in_training
0,zero_width_space,1669,0.8573,0.4825,0.0321,True
1,paraphrase,1662,0.9084,0.8700,0.2272,True
2,homoglyph,1708,0.9273,0.6318,0.0194,True
3,insert_paragraphs,1612,0.9918,0.9771,0.0969,True
4,upper_lower,1675,0.9957,0.9696,0.0171,True
5,none,1476,0.9958,0.9883,0.0453,True
6,perplexity_misspelling,1601,0.9959,0.9855,0.0349,True
7,synonym,1668,0.9961,0.9768,0.0153,True
8,number,1592,0.9965,0.9810,0.0412,True
9,whitespace,1682,0.9968,0.9817,0.0197,True



--- Robust-all roberta-base (all attacks; RAID numbers are in-distribution): slice by generator (clean only) ---


,model,n,AUROC,TPR@thr,FPR@thr
0,human,706,NaN,NaN,0.0453
1,chatgpt,33,0.9994,1.0000,NaN
2,cohere,42,0.9784,0.9286,NaN
3,cohere-chat,49,0.9990,1.0000,NaN
4,gpt2,87,0.9985,1.0000,NaN
5,gpt3,49,1.0000,1.0000,NaN
6,gpt4,48,0.9998,1.0000,NaN
7,llama-chat,95,0.9999,1.0000,NaN
8,mistral,106,0.9969,0.9717,NaN
9,mistral-chat,91,0.9998,1.0000,NaN



--- Robust-all roberta-base (all attacks; RAID numbers are in-distribution): slice by domain (all attacks) ---


,domain,n,AUROC,TPR@thr,FPR@thr
0,wiki,2701,0.9629,0.8592,0.0647
1,reviews,1423,0.9698,0.9095,0.1087
2,books,2336,0.9704,0.8914,0.0720
3,news,2618,0.9754,0.8837,0.0528
4,reddit,2626,0.9798,0.8934,0.0488
5,poetry,2299,0.9892,0.9179,0.0316
6,abstracts,2818,0.9898,0.8899,0.0398
7,recipes,2821,0.9969,0.9564,0.0136


## 5. Final Comparison (static RAID attacks)
All three models side by side, per attack, at their own 5%-FPR thresholds (calibrated on clean human validation text). Rows are sorted from the attack that hurts the naive model most.

In [11]:
cmp = side_by_side({GPT2_NAME: rep_gpt2, NAIVE_NAME: rep_naive, ROBUST_NAME: rep_robust,
                    ROBUST_ALL_NAME: rep_all}, sort_by=NAIVE_NAME)
display(cmp.round(3))

# Sanity check: FPR on clean human TEST text for each model (should be ~5%)
print("\n--- FPR on clean human TEST text (calibrated target = 5%) ---")
for name, pdf, thr in [(GPT2_NAME, test_scores, thr_gpt2), (NAIVE_NAME, rt_naive, thr_naive),
                       (ROBUST_NAME, rt_robust, thr_robust), (ROBUST_ALL_NAME, rt_all, thr_all)]:
    h = pdf[(pdf.label == 0) & (pdf.attack == "none")].score
    print(f"{name:30s} thr = {thr:.6f}   FPR = {(h > thr).mean() * 100:.1f}%  (n={len(h)})")

GPT-2 zero-shot                 Naive roberta-base  \
                                 AUROC TPR@thr FPR@thr              AUROC   
attack                                                                      
zero_width_space                 0.490   0.996   1.000              0.596   
paraphrase                       0.690   0.147   0.023              0.773   
homoglyph                        0.532   0.031   0.002              0.807   
insert_paragraphs                0.755   0.370   0.036              0.886   
upper_lower                      0.745   0.184   0.009              0.995   
whitespace                       0.737   0.115   0.001              0.995   
none                             0.756   0.381   0.042              0.996   
synonym                          0.763   0.159   0.002              0.996   
perplexity_misspelling           0.753   0.318   0.025              0.996   
number                           0.737   0.337   0.015              0.996   
alternative_spelling             0.733   0.336   0.038              0.997   
article_deletion                 0.731   0.235   0.019              0.997   

                                       Robust roberta-base                  \
                       TPR@thr FPR@thr               AUROC TPR@thr FPR@thr   
attack                                                                       
zero_width_space         1.000   1.000               0.617   0.180   0.015   
paraphrase               0.677   0.238               0.823   0.592   0.160   
homoglyph                0.886   0.570               0.884   0.568   0.018   
insert_paragraphs        0.999   0.936               0.972   0.898   0.052   
upper_lower              0.959   0.019               0.984   0.911   0.038   
whitespace               0.973   0.027               0.984   0.919   0.020   
none                     0.986   0.058               0.988   0.932   0.040   
synonym                  0.983   0.019               0.992   0.932   0.020   
perplexity_misspelling   0.979   0.037               0.988   0.935   0.037   
number                   0.987   0.044               0.987   0.927   0.036   
alternative_spelling     0.988   0.039               0.992   0.949   0.031   
article_deletion         0.986   0.028               0.989   0.940   0.030   

                       Robust-all roberta-base                  
                                         AUROC TPR@thr FPR@thr  
attack                                                          
zero_width_space                         0.857   0.482   0.032  
paraphrase                               0.908   0.870   0.227  
homoglyph                                0.927   0.632   0.019  
insert_paragraphs                        0.992   0.977   0.097  
upper_lower                              0.996   0.970   0.017  
whitespace                               0.997   0.982   0.020  
none                                     0.996   0.988   0.045  
synonym                                  0.996   0.977   0.015  
perplexity_misspelling                   0.996   0.985   0.035  
number                                   0.997   0.981   0.041  
alternative_spelling                     0.998   0.987   0.029  
article_deletion                         0.997   0.977   0.024


--- FPR on clean human TEST text (calibrated target = 5%) ---
GPT-2 zero-shot                thr = -2.378963   FPR = 4.2%  (n=706)
Naive roberta-base             thr = 0.424514   FPR = 5.8%  (n=706)
Robust roberta-base            thr = 0.966998   FPR = 4.0%  (n=706)
Robust-all roberta-base        thr = 0.938828   FPR = 4.5%  (n=706)


## 6. MAGE: out-of-distribution check
Testing whether the models generalize to an entirely unseen dataset with different generators and domains. Thresholds are the RAID-calibrated 5%-FPR ones, so a shifted FPR here is itself informative.

In [13]:

# ============================================================
# 6. MAGE OUT-OF-DISTRIBUTION EVALUATION
# Evaluate on BOTH standard GPT OOD and GPT paraphrase OOD
# All available rows are used (no random sampling).
# ============================================================

from huggingface_hub import hf_hub_download

MAGE_FILES = {
    "MAGE GPT-4 OOD": "test_ood_set_gpt.csv",
    "MAGE GPT-4 Paraphrase OOD": "test_ood_set_gpt_para.csv"
}

MODELS = {
    GPT2_NAME: (detector_score, thr_gpt2),
    NAIVE_NAME: (lambda x: roberta_scores(clf_naive, x), thr_naive),
    ROBUST_NAME: (lambda x: roberta_scores(clf_robust, x), thr_robust),
    ROBUST_ALL_NAME: (lambda x: roberta_scores(clf_all, x), thr_all)
}


def load_mage(filename):
    path = hf_hub_download(
        "yaful/MAGE",
        filename,
        repo_type="dataset"
    )

    data = pd.read_csv(path)

    # Handle files without a header
    if not {"text", "label"}.issubset(data.columns):
        data = pd.read_csv(path, header=None)
        data = data.iloc[:, :3]
        cols = ["text", "label", "src"][:len(data.columns)]
        data.columns = cols

    # Remove missing or empty text
    data = data.dropna(subset=["text", "label"]).copy()
    data["text"] = data["text"].astype(str).str.strip()
    data = data[data["text"].str.len() > 0].reset_index(drop=True)

    data["label"] = pd.to_numeric(data["label"])

    # MAGE: 0 = AI, 1 = Human
    # RAID: 1 = AI, 0 = Human
    data["label_ai"] = 1 - data["label"]

    assert set(data["label_ai"].unique()).issubset({0, 1})

    return data


def evaluate_mage(data, dataset_name):

    texts = data["text"].tolist()
    y = data["label_ai"].to_numpy()

    print(f"\nScoring {dataset_name}: {len(data):,} examples")

    results = []
    domain_results = []

    for model_name, (score_fn, threshold) in MODELS.items():

        scores = np.asarray(score_fn(texts))

        assert len(scores) == len(y)

        # Overall evaluation
        results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "N": len(y),
            "N_AI": int((y == 1).sum()),
            "N_Human": int((y == 0).sum()),
            "AUROC": roc_auc_score(y, scores),
            "TPR@thr": (scores[y == 1] > threshold).mean(),
            "FPR@thr": (scores[y == 0] > threshold).mean()
        })

        # Evaluation by source/domain
        if "src" in data.columns:

            for source, indices in data.groupby("src").groups.items():

                idx = np.asarray(list(indices))
                ys = y[idx]
                ss = scores[idx]

                domain_results.append({
                    "Dataset": dataset_name,
                    "Source": source,
                    "Model": model_name,
                    "N": len(idx),
                    "N_AI": int((ys == 1).sum()),
                    "N_Human": int((ys == 0).sum()),
                    "AUROC": (
                        roc_auc_score(ys, ss)
                        if len(np.unique(ys)) == 2
                        else np.nan
                    ),
                    "TPR@thr": (
                        (ss[ys == 1] > threshold).mean()
                        if (ys == 1).any() else np.nan
                    ),
                    "FPR@thr": (
                        (ss[ys == 0] > threshold).mean()
                        if (ys == 0).any() else np.nan
                    )
                })

    return pd.DataFrame(results), pd.DataFrame(domain_results)


# Run evaluation on both MAGE datasets
mage_overall = []
mage_domains = []
mage_data_counts = []

for dataset_name, filename in MAGE_FILES.items():

    print("\n" + "=" * 75)
    print(dataset_name)
    print("=" * 75)

    data = load_mage(filename)

    print("Dataset shape:", data.shape)
    print("Label counts (original MAGE):",
          data["label"].value_counts().to_dict())

    if "src" in data.columns:
        print("\nRows per source/domain:")
        display(data["src"].value_counts().to_frame("Rows"))

    print("\nHuman vs AI:")
    print(data["label_ai"].value_counts().rename(
        index={0: "Human", 1: "AI"}
    ))

    mage_data_counts.append({
        "Dataset": dataset_name,
        "Total": len(data),
        "AI": int((data["label_ai"] == 1).sum()),
        "Human": int((data["label_ai"] == 0).sum())
    })

    overall, domains = evaluate_mage(data, dataset_name)

    mage_overall.append(overall)
    if not domains.empty:
        mage_domains.append(domains)

    del data


# ============================================================
# FINAL RESULTS
# ============================================================

mage_overall_df = pd.concat(mage_overall, ignore_index=True)

print("\n" + "=" * 75)
print("MAGE OVERALL COMPARISON")
print("=" * 75)

display(
    mage_overall_df.sort_values(
        ["Dataset", "AUROC"],
        ascending=[True, False]
    ).round(4)
)

print("\nMAGE DATASET SIZES:")
display(pd.DataFrame(mage_data_counts))


# Domain-wise comparison
if mage_domains:

    mage_domain_df = pd.concat(mage_domains, ignore_index=True)

    print("\n" + "=" * 75)
    print("MAGE SOURCE / DOMAIN-WISE COMPARISON")
    print("=" * 75)

    display(
        mage_domain_df.sort_values(
            ["Dataset", "Source", "AUROC"],
            ascending=[True, True, False]
        ).round(4)
    )



MAGE GPT-4 OOD


test_ood_set_gpt.csv:   0%|          | 0.00/2.19M [00:00<?, ?B/s]

Dataset shape: (1562, 4)
Label counts (original MAGE): {0: 800, 1: 762}

Rows per source/domain:


,Rows
src,
cnn_human,200
cnn_gpt4,200
dialogsum_gpt4,200
imdb_human,200
pubmed_human,200
imdb_gpt4,200
pubmed_gpt4,200
dialogsum_human,162



Human vs AI:
label_ai
AI       800
Human    762
Name: count, dtype: int64

Scoring MAGE GPT-4 OOD: 1,562 examples

MAGE GPT-4 Paraphrase OOD
Dataset shape: (2362, 4)
Label counts (original MAGE): {0: 1600, 1: 762}

Rows per source/domain:


,Rows
src,
cnn_human_para,200
cnn_gpt4_para,200
dialogsum_human_para,200
dialogsum_gpt4_para,200
imdb_human_para,200
imdb_gpt4_para,200
pubmed_human_para,200
pubmed_gpt4_para,200
cnn_human,200


... (2 outputs left out)


MAGE DATASET SIZES:


,Dataset,Total,AI,Human
0,MAGE GPT-4 OOD,1562,800,762
1,MAGE GPT-4 Paraphrase OOD,2362,1600,762



MAGE SOURCE / DOMAIN-WISE COMPARISON


,Dataset,Source,Model,N,N_AI,N_Human,AUROC,TPR@thr,FPR@thr
0,MAGE GPT-4 OOD,cnn_gpt4,GPT-2 zero-shot,200,200,0,NaN,0.035,NaN
8,MAGE GPT-4 OOD,cnn_gpt4,Naive roberta-base,200,200,0,NaN,0.925,NaN
16,MAGE GPT-4 OOD,cnn_gpt4,Robust roberta-base,200,200,0,NaN,0.790,NaN
24,MAGE GPT-4 OOD,cnn_gpt4,Robust-all roberta-base,200,200,0,NaN,0.985,NaN
1,MAGE GPT-4 OOD,cnn_human,GPT-2 zero-shot,200,0,200,NaN,NaN,0.01
...,...,...,...,...,...,...,...,...,...
78,MAGE GPT-4 Paraphrase OOD,pubmed_human,Robust-all roberta-base,200,0,200,NaN,NaN,0.47
43,MAGE GPT-4 Paraphrase OOD,pubmed_human_para,GPT-2 zero-shot,200,200,0,NaN,0.005,NaN
55,MAGE GPT-4 Paraphrase OOD,pubmed_human_para,Naive roberta-base,200,200,0,NaN,0.930,NaN
67,MAGE GPT-4 Paraphrase OOD,pubmed_human_para,Robust roberta-base,200,200,0,NaN,0.605,NaN
